In [2]:
import pandas as pd

fake = pd.read_csv("/content/Fake.csv")
real = pd.read_csv("/content/True.csv")

fake["label"] = 1
real["label"] = 0

data = pd.concat([fake, real])

data = data[["text", "label"]]

data.head()

,text,label
0,Donald Trump just couldn t wish all Americans ...,1
1,House Intelligence Committee Chairman Devin Nu...,1
2,"On Friday, it was revealed that former Milwauk...",1
3,"On Christmas day, Donald Trump announced that ...",1
4,Pope Francis used his annual Christmas Day mes...,1


In [4]:
from sklearn.model_selection import train_test_split

X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

nb = MultinomialNB()

lr = LogisticRegression(max_iter=1000)

voting = VotingClassifier(
    estimators=[
        ("nb", nb),
        ("lr", lr)
    ],
    voting="soft"
)

rf = RandomForestClassifier(n_estimators=50)

adb = AdaBoostClassifier(n_estimators=50)

In [8]:
nb.fit(X_train_tfidf, y_train)

lr.fit(X_train_tfidf, y_train)

voting.fit(X_train_tfidf, y_train)

rf.fit(X_train_tfidf, y_train)

adb.fit(X_train_tfidf, y_train)

AdaBoostClassifier()

In [9]:
from sklearn.metrics import accuracy_score

models = {
    "Naive Bayes": nb,
    "Logistic Regression": lr,
    "Voting": voting,
    "Random Forest": rf,
    "AdaBoost": adb
}

for name, model in models.items():

    pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, pred)

    print(name, ":", acc)

Naive Bayes : 0.9287305122494433
Logistic Regression : 0.9864142538975501
Voting : 0.96815144766147
Random Forest : 0.9975501113585746
AdaBoost : 0.9938752783964365


In [10]:
import gradio as gr

def detect_news(news):

    vec = vectorizer.transform([news])

    pred = voting.predict(vec)[0]

    if pred == 1:
        return "Fake News"
    else:
        return "Real News"

app = gr.Interface(
    fn=detect_news,

    inputs=gr.Textbox(lines=6, placeholder="Enter news text here..."),

    outputs="text",

    title="Fake News Detection using Ensemble Learning",

    description="Detect whether a news article is Fake or Real."
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b66f19691aae21b56f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
import joblib
import os

# Create the directory if it doesn't exist
output_dir = "/content/model"
os.makedirs(output_dir, exist_ok=True)

# Save the vectorizer
joblib.dump(vectorizer, os.path.join(output_dir, "vectorizer.pkl"))

# Save individual models
joblib.dump(nb, os.path.join(output_dir, "naive_bayes_model.pkl"))
joblib.dump(lr, os.path.join(output_dir, "logistic_regression_model.pkl"))
joblib.dump(voting, os.path.join(output_dir, "voting_classifier_model.pkl"))
joblib.dump(rf, os.path.join(output_dir, "random_forest_model.pkl"))
joblib.dump(adb, os.path.join(output_dir, "adaboost_model.pkl"))

print(f"Models and vectorizer saved to {output_dir}")


Models and vectorizer saved to /content/model


In [12]:
import joblib
import gradio as gr
import os

# Define the path where models are saved
output_dir = "/content/model"

# Load the vectorizer and the voting classifier
vectorizer = joblib.load(os.path.join(output_dir, "vectorizer.pkl"))
voting = joblib.load(os.path.join(output_dir, "voting_classifier_model.pkl"))

def detect_news(news):
    vec = vectorizer.transform([news])
    pred = voting.predict(vec)[0]

    if pred == 1:
        return "Fake News"
    else:
        return "Real News"

# Create and launch the Gradio interface
app = gr.Interface(
    fn=detect_news,
    inputs=gr.Textbox(lines=6, placeholder="Enter news text here..."),
    outputs="text",
    title="Fake News Detection using Ensemble Learning",
    description="Detect whether a news article is Fake or Real using pre-trained models."
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://501644864246b2f1c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
